In [ ]:
import pandas as pd
from pathlib import Path

DOWNLOADS = Path.home() / "Downloads"
stress = pd.read_csv(DOWNLOADS / "stress_weeklies_aug2024nov2025.csv")
list(stress.columns)

In [ ]:
ID_COL = "name"
DATE_RANGE_COL = "date_range"
STRESS_COL = "total_stress_high"

In [ ]:
# Split date_range into start and end
stress[["week_start", "week_end"]] = (stress[DATE_RANGE_COL].str.split(" to ", expand=True))

# Convert to datetime
stress["week_start"] = pd.to_datetime(stress["week_start"])
stress["week_end"] = pd.to_datetime(stress["week_end"])

stress.head()

In [ ]:
print("participants:", stress[ID_COL].nunique())
print("participant-weeks:", stress.shape[0]) #number of rows aka weeks

stress[STRESS_COL].describe()

In [ ]:
sleep = pd.read_csv(DOWNLOADS / "sleepsd_weeklies_aug2024nov2025.csv")
list(sleep.columns)

In [ ]:
sleep[["week_start", "week_end"]] = (sleep["date_range"].str.split(" to ", expand=True))

sleep["week_start"] = pd.to_datetime(sleep["week_start"])
sleep["week_end"] = pd.to_datetime(sleep["week_end"])

sleep.head()

In [ ]:
SLEEPSTART_COL = "SD_bedtime_start"

In [ ]:
stress_sleep = stress.merge(sleep[[ID_COL, "week_start", "week_end", SLEEPSTART_COL]],on=[ID_COL, "week_start", "week_end"],how="left")

stress_sleep.shape

In [ ]:
stress_sleep[[STRESS_COL, SLEEPSTART_COL]].isna().mean()

In [ ]:
exercise = pd.read_csv(DOWNLOADS / "exercise_weeklies_aug2024nov2025.csv")
list(exercise.columns)

In [ ]:
# parse date_range for exercise
exercise[["week_start", "week_end"]] = (exercise["date_range"].str.split(" to ", expand=True))
exercise["week_start"] = pd.to_datetime(exercise["week_start"])
exercise["week_end"] = pd.to_datetime(exercise["week_end"])

# exercise metric column
EXERCISECAL_COL = "active_calories"  # could use a different metric from columns in exercise

# merge into stress_sleep df
stress_sleep_ex = stress_sleep.merge(exercise[[ID_COL, "week_start", "week_end", EXERCISECAL_COL]],on=[ID_COL, "week_start", "week_end"],how="left")

stress_sleep_ex.shape

In [ ]:
# inspect a few date ranges side by side
stress[[ID_COL, "week_start", "week_end"]].head(5)
exercise[[ID_COL, "week_start", "week_end"]].head(5)

In [ ]:
stress_sleep_ex[[STRESS_COL, SLEEPSTART_COL, EXERCISECAL_COL]].isna().mean()

In [ ]:
exercise_person = (
    exercise
    .groupby(ID_COL)[EXERCISECAL_COL]
    .mean()
    .reset_index()
    .rename(columns={EXERCISECAL_COL: "mean_active_calories"})
)

stress_sleep_ex = stress_sleep.merge(exercise_person,on=ID_COL,how="left")

stress_sleep_ex[[STRESS_COL, SLEEPSTART_COL, "mean_active_calories"]].isna().mean()

In [ ]:
stress_sleep["SD_bedtime_start_td"] = pd.to_timedelta(stress_sleep[SLEEPSTART_COL])

stress_sleep["SD_bedtime_start_min"] = (stress_sleep["SD_bedtime_start_td"].dt.total_seconds() / 60)

stress_sleep["SD_bedtime_start_min"].describe()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# stress histogram
plt.figure()
stress_sleep[STRESS_COL].dropna().hist(bins=40)
plt.title(f"{STRESS_COL} (weekly)")
plt.xlabel(STRESS_COL)
plt.ylabel("count")
plt.tight_layout()
plt.savefig(OUT_DIR / f"hist_{STRESS_COL}.png", dpi=200)
plt.show()

# sleep SD histogram (minutes) with outlier-safe x-axis
sleep_min = stress_sleep["SD_bedtime_start_min"].dropna()

# cap x-axis for visualization so the max=719 min doesn't flatten everything
x_cap = sleep_min.quantile(0.99)  # data-driven cap

plt.figure()
sleep_min.hist(bins=40, range=(0, x_cap))
plt.title(f"SD_bedtime_start (minutes), capped at 99th pct = {x_cap:.1f}")
plt.xlabel("minutes")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(OUT_DIR / "hist_SD_bedtime_start_min_capped.png", dpi=200)
plt.show()

print("Sleep minutes cap (99th percentile):", round(x_cap, 2))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DOWNLOADS = Path.home() / "Downloads"

# list candidate exports
cands = [p for p in DOWNLOADS.glob("*.csv") if "week" in p.name.lower() or "Jan26" in p.name or "weeklies" in p.name.lower()]
[c.name for c in cands][:20]

In [ ]:
weeklies_fp = DOWNLOADS / "StudyOnTypicallyIgno-Jan26weeklies_DATA_2026-01-04_0526.csv"  
weeklies = pd.read_csv(weeklies_fp)

weeklies.shape, list(weeklies.columns)[:40]

In [ ]:
ID_COL = "id_num"

In [ ]:
# cycle starts from menzie_start
cycles = weeklies[[ID_COL, "menzie_start"]].copy()

# parse dates safely
cycles["menzie_start"] = pd.to_datetime(cycles["menzie_start"], errors="coerce")
cycles["menzie_start"].dtype

# drop rows without a cycle start
cycles = cycles.dropna(subset=["menzie_start"])

# sort within participant
cycles = cycles.sort_values([ID_COL, "menzie_start"])

cycles.head()

In [ ]:
cycles.groupby(ID_COL).size().describe()

In [ ]:
cycles["menzie_start"].describe()

In [ ]:
cycles["menzie_start"] = pd.to_datetime(cycles["menzie_start"],errors="coerce",utc=False)

cycles["menzie_start"].dtype

In [ ]:
cycles.dtypes

In [ ]:
import pandas.api.types as ptypes
ptypes.is_datetime64_any_dtype(cycles["menzie_start"])

In [ ]:
# sorted
cycles = cycles.sort_values([ID_COL, "menzie_start"])

# cycle length in days
cycles["cycle_length_days"] = (cycles.groupby(ID_COL)["menzie_start"].diff().dt.days)

# drop first start per person so no previous cycle
cycles2 = cycles.dropna(subset=["cycle_length_days"]).copy()

cycles2["cycle_length_days"].describe()

In [ ]:
cycles2["cycle_length_days"].min(), cycles2["cycle_length_days"].max()

In [ ]:
(cycles2["cycle_length_days"] < 10).mean(), (cycles2["cycle_length_days"] > 90).mean()

In [ ]:
cycles_nodup = cycles.drop_duplicates(subset=[ID_COL, "menzie_start"]).copy()
cycles_nodup = cycles_nodup.sort_values([ID_COL, "menzie_start"])

cycles_nodup["cycle_length_days"] = (cycles_nodup.groupby(ID_COL)["menzie_start"].diff().dt.days)

cycles2 = cycles_nodup.dropna(subset=["cycle_length_days"]).copy()
cycles2["cycle_length_days"].describe()

In [ ]:
(cycles2["cycle_length_days"] < 10).mean(), (cycles2["cycle_length_days"] > 90).mean()

In [ ]:
cycles2_qc = cycles2.query("cycle_length_days >= 10 and cycle_length_days <= 90").copy()

cycles2_qc["cycle_length_days"].describe()

In [ ]:
cycle_reg = (cycles2_qc.groupby(ID_COL)["cycle_length_days"].agg(mean_cycle_length="mean", sd_cycle_length="std", n_cycles="count").reset_index())

# participants with enough cycles to estimate SD
cycle_reg = cycle_reg[cycle_reg["n_cycles"] >= 3].copy()

cycle_reg.describe()
cycle_reg.head()

In [ ]:
stress_person = (stress_sleep.groupby("name")[STRESS_COL].mean().reset_index().rename(columns={STRESS_COL: "mean_total_stress_high"}))

stress_person.head()

In [ ]:
# stress_person uses id_num
stress_person = stress_person.rename(columns={"name": "id_num"}).copy()

stress_person["id_num"] = pd.to_numeric(stress_person["id_num"], errors="coerce")

stress_person.head()
stress_person["id_num"].isna().mean()

In [ ]:
cycle_reg.columns

In [ ]:
bg_fp = DOWNLOADS / "cleaned_backgrounddata.csv"
background = pd.read_csv(bg_fp)

background.shape
background.columns

In [ ]:
AGE_COL = "years"
HEIGHT_COL = "height" #inches
WEIGHT_COL = "weight" #pounds

background[HEIGHT_COL] = pd.to_numeric(background[HEIGHT_COL], errors="coerce")
background[WEIGHT_COL] = pd.to_numeric(background[WEIGHT_COL], errors="coerce")

background["bmi"] = (background[WEIGHT_COL] / (background[HEIGHT_COL]**2)) * 703
BMI_COL = "bmi"

background[[AGE_COL, BMI_COL]].describe()

In [ ]:
background[HEIGHT_COL].describe(), background[WEIGHT_COL].describe()

In [ ]:
# covariates table
covars = (background[["id_num", AGE_COL, BMI_COL]].copy())

covars["id_num"] = pd.to_numeric(covars["id_num"], errors="coerce")
covars = covars.dropna(subset=["id_num"])
covars = covars.drop_duplicates(subset=["id_num"])

covars.head()

In [ ]:
# force consistent merge key dtype
for df, name in [(cycle_reg, "cycle_reg"), (stress_person, "stress_person"), (covars, "covars")]:df["id_num"] = pd.to_numeric(df["id_num"], errors="coerce")
print(name, df["id_num"].dtype, df["id_num"].isna().mean())

In [ ]:
# regression dataframe
df_reg = (cycle_reg.merge(stress_person, on="id_num", how="left").merge(covars, on="id_num", how="left"))

df_reg.shape
df_reg.head()

In [ ]:
# missingness
df_reg[["sd_cycle_length","mean_total_stress_high",AGE_COL,BMI_COL]].isna().mean()

In [ ]:
dfm = df_reg.dropna(subset=["sd_cycle_length", "mean_total_stress_high", AGE_COL, BMI_COL]).copy()

dfm.shape

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(f"sd_cycle_length ~ mean_total_stress_high + {AGE_COL} + {BMI_COL}",data=dfm).fit()

print(model.summary())

In [ ]:
# z-score helpers
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

# z-score predictors
dfm = dfm.copy()
dfm["stress_z"] = zscore(dfm["mean_total_stress_high"])
dfm["age_z"]    = zscore(dfm[AGE_COL])
dfm["bmi_z"]    = zscore(dfm[BMI_COL])

dfm[["stress_z", "age_z", "bmi_z"]].describe()

In [ ]:
dfm[["stress_z", "age_z", "bmi_z"]].agg(["mean","std"])

In [ ]:
import statsmodels.formula.api as smf

model_z = smf.ols(
    "sd_cycle_length ~ stress_z + age_z + bmi_z",
    data=dfm
).fit()

print(model_z.summary())

In [ ]:
print("SLEEPSTART_COL =", SLEEPSTART_COL)
stress_sleep[SLEEPSTART_COL].head(10), stress_sleep[SLEEPSTART_COL].dtype

In [ ]:
# make a numeric minutes column from SLEEPSTART_COL
sleep_raw = stress_sleep[SLEEPSTART_COL]

if np.issubdtype(sleep_raw.dtype, np.number):
    # already numeric
    stress_sleep["sleepstart_sd_min"] = sleep_raw
else:
    stress_sleep["sleepstart_sd_min"] = (
        pd.to_timedelta(sleep_raw, errors="coerce").dt.total_seconds() / 60)

stress_sleep["sleepstart_sd_min"].describe()
stress_sleep[["name", SLEEPSTART_COL, "sleepstart_sd_min"]].head(10)

In [ ]:
sleep_person = (stress_sleep.groupby("name")["sleepstart_sd_min"].mean().reset_index()
    .rename(columns={
        "name": "id_num",
        "sleepstart_sd_min": "mean_sleepstart_sd_min"
    })
)

sleep_person["id_num"] = pd.to_numeric(sleep_person["id_num"], errors="coerce")
sleep_person.head()
sleep_person.shape

In [ ]:
df_reg_sleep = (cycle_reg.merge(sleep_person, on="id_num", how="left").merge(covars, on="id_num", how="left"))

df_reg_sleep[["sd_cycle_length", "mean_sleepstart_sd_min", AGE_COL, BMI_COL]].isna().mean()

dfm_sleep = df_reg_sleep.dropna(subset=["sd_cycle_length", "mean_sleepstart_sd_min", AGE_COL, BMI_COL]).copy()

# z-score
dfm_sleep["sleep_z"] = zscore(dfm_sleep["mean_sleepstart_sd_min"])
dfm_sleep["age_z"]   = zscore(dfm_sleep[AGE_COL])
dfm_sleep["bmi_z"]   = zscore(dfm_sleep[BMI_COL])

model_sleep_z = smf.ols(
    "sd_cycle_length ~ sleep_z + age_z + bmi_z",
    data=dfm_sleep
).fit()

print(model_sleep_z.summary())

In [ ]:
# EXERCISE

exercise = pd.read_csv(DOWNLOADS / "exercise_weeklies_aug2024nov2025.csv")
exercise.columns

In [ ]:
exercise["name"] = pd.to_numeric(exercise["name"], errors="coerce")

exercise_person = (exercise.groupby("name")["active_calories"].mean().reset_index().rename(columns={
        "name": "id_num",
        "active_calories": "mean_active_calories"
    })
)

exercise_person.head()
exercise_person.shape
exercise_person["mean_active_calories"].describe()

In [ ]:
exercise_person = (exercise.groupby("name")["active_calories"].agg(mean_active_calories="mean", n_weeks_ex="count").reset_index().rename(columns={"name": "id_num"}))

exercise_person["id_num"] = pd.to_numeric(exercise_person["id_num"], errors="coerce")

exercise_person[["n_weeks_ex","mean_active_calories"]].describe()

In [ ]:
df_reg_ex = (cycle_reg.merge(exercise_person, on="id_num", how="left").merge(covars, on="id_num", how="left"))

df_reg_ex[["sd_cycle_length","mean_active_calories", AGE_COL, BMI_COL]].isna().mean()

dfm_ex = df_reg_ex.dropna(subset=["sd_cycle_length","mean_active_calories", AGE_COL, BMI_COL]).copy()

dfm_ex["exercise_z"] = zscore(dfm_ex["mean_active_calories"])
dfm_ex["age_z"]      = zscore(dfm_ex[AGE_COL])
dfm_ex["bmi_z"]      = zscore(dfm_ex[BMI_COL])

model_ex = smf.ols("sd_cycle_length ~ exercise_z + age_z + bmi_z",data=dfm_ex).fit()

print(model_ex.summary())

In [ ]:
# stress regression plot (partial regression adjusted for age and BMI) 
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

# stress model dataframe = dfm
y_col = "sd_cycle_length"
x_col = "stress_z"
covars = ["age_z", "bmi_z"]

# residualize y and x on covariates
Xc = sm.add_constant(dfm[covars])
y_res = pd.Series(sm.OLS(dfm[y_col], Xc, missing="drop").fit().resid, index=dfm.index)
x_res = pd.Series(sm.OLS(dfm[x_col], Xc, missing="drop").fit().resid, index=dfm.index)

# fit line on residuals
line = sm.OLS(y_res, sm.add_constant(x_res)).fit()
intercept = float(line.params.iloc[0])
slope = float(line.params.iloc[1])

# plot
plt.figure(figsize=(5, 4))
plt.scatter(x_res, y_res, alpha=0.7)
xs = np.linspace(x_res.min(), x_res.max(), 200)
plt.plot(xs, intercept + slope * xs)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("stress_z (adjusted for age + BMI)")
plt.ylabel("sd_cycle_length (adjusted for age + BMI)")
plt.title("Stress vs cycle irregularity\n(partial regression)")
plt.tight_layout()
plt.show()

In [ ]:
# sleep regression plot (partial regression adjusted for age and BMI)
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

# sleep model dataframe = dfm_sleep
y_col = "sd_cycle_length"
x_col = "sleep_z"
covars = ["age_z", "bmi_z"]

# residualize y and x on covariates
Xc = sm.add_constant(dfm_sleep[covars])
y_res = pd.Series(
    sm.OLS(dfm_sleep[y_col], Xc, missing="drop").fit().resid,
    index=dfm_sleep.index
)
x_res = pd.Series(
    sm.OLS(dfm_sleep[x_col], Xc, missing="drop").fit().resid,
    index=dfm_sleep.index
)

# fit line on residuals
line = sm.OLS(y_res, sm.add_constant(x_res)).fit()
intercept = float(line.params.iloc[0])
slope = float(line.params.iloc[1])

# plot
plt.figure(figsize=(5, 4))
plt.scatter(x_res, y_res, alpha=0.7)
xs = np.linspace(x_res.min(), x_res.max(), 200)
plt.plot(xs, intercept + slope * xs)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("sleep_z (adjusted for age + BMI)")
plt.ylabel("sd_cycle_length (adjusted for age + BMI)")
plt.title("Sleep variability vs cycle irregularity\n(partial regression)")
plt.tight_layout()
plt.show()

In [ ]:
# exercise regression plot (partial regression adjusted for age and BMI)
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

# exercise model dataframe = dfm_ex
y_col = "sd_cycle_length"
x_col = "exercise_z"
covars = ["age_z", "bmi_z"]

# residualize y and x on covariates
Xc = sm.add_constant(dfm_ex[covars])
y_res = pd.Series(
    sm.OLS(dfm_ex[y_col], Xc, missing="drop").fit().resid,
    index=dfm_ex.index
)
x_res = pd.Series(
    sm.OLS(dfm_ex[x_col], Xc, missing="drop").fit().resid,
    index=dfm_ex.index
)

# fit line on residuals
line = sm.OLS(y_res, sm.add_constant(x_res)).fit()
intercept = float(line.params.iloc[0])
slope = float(line.params.iloc[1])

# plot
plt.figure(figsize=(5, 4))
plt.scatter(x_res, y_res, alpha=0.7)
xs = np.linspace(x_res.min(), x_res.max(), 200)
plt.plot(xs, intercept + slope * xs)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("exercise_z (adjusted for age + BMI)")
plt.ylabel("sd_cycle_length (adjusted for age + BMI)")
plt.title("Exercise vs cycle irregularity\n(partial regression)")
plt.tight_layout()
plt.show()